# Laboratorio: síntesis inversa de un mecanismo de cuatro barras

Este cuaderno recibe una curva cerrada dibujada, calcula las longitudes **L1, L2, L3 y L4** y la posición del punto trazador **P**, compara ambas trayectorias y anima el mecanismo.

**Flujo:** ejecutar las celdas en orden → dibujar → optimizar → registrar resultados → construir con palitos de paleta.

> Todo el cálculo se realiza en esta sesión. No se envía la curva a servicios externos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import pandas as pd, math, time

np.set_printoptions(precision=3, suppress=True)
print('Entorno preparado.')

## 1. Dibujar la curva

Ejecuta la celda y dibuja con el mouse o lápiz. Pulsa **FINALIZAR CURVA** cuando el contorno esté cerrado. Evita cruces y trazos demasiado pequeños.

In [ ]:
from google.colab import output

canvas_html = r'''new Promise((resolve) => {
  const box=document.createElement('div');
  box.innerHTML=`<div style="font-family:Arial"><canvas id="curveCanvas" width="760" height="430" style="border:2px solid #12304a;border-radius:12px;background:white;touch-action:none"></canvas><br><button id="finishCurve" style="margin-top:8px;padding:10px 20px;background:#167d9a;color:white;border:0;border-radius:8px;font-weight:bold">FINALIZAR CURVA</button> <button id="clearCurve" style="padding:10px 20px">BORRAR</button></div>`;
  document.body.appendChild(box);
  const c=box.querySelector('#curveCanvas'), ctx=c.getContext('2d'); let pts=[], drawing=false;
  function point(e){const r=c.getBoundingClientRect();const q=e.touches?e.touches[0]:e;return [(q.clientX-r.left)*c.width/r.width,(q.clientY-r.top)*c.height/r.height]}
  function start(e){e.preventDefault();drawing=true;pts=[point(e)];ctx.beginPath();ctx.moveTo(...pts[0])}
  function move(e){if(!drawing)return;e.preventDefault();const p=point(e);const last=pts[pts.length-1];if(Math.hypot(p[0]-last[0],p[1]-last[1])>2){pts.push(p);ctx.lineTo(...p);ctx.strokeStyle='#12304a';ctx.lineWidth=4;ctx.lineCap='round';ctx.stroke()}}
  function stop(){drawing=false}
  c.addEventListener('pointerdown',start);c.addEventListener('pointermove',move);c.addEventListener('pointerup',stop);c.addEventListener('pointerleave',stop);
  box.querySelector('#clearCurve').onclick=()=>{pts=[];ctx.clearRect(0,0,c.width,c.height)};
  box.querySelector('#finishCurve').onclick=()=>{if(pts.length<20){alert('Dibuja una curva más larga');return}box.remove();resolve(pts)};
})'''

curve_raw = np.asarray(output.eval_js(canvas_html), dtype=float)
curve_raw[:,1] *= -1  # eje matemático hacia arriba
print(f'{len(curve_raw)} muestras capturadas')
plt.figure(figsize=(9,4)); plt.plot(curve_raw[:,0],curve_raw[:,1],lw=3);plt.axis('equal');plt.grid(alpha=.2);plt.title('Curva dibujada');plt.show()

## 2. Preparación geométrica

Se eliminan puntos repetidos, se cierra el contorno y se remuestrea por longitud de arco. La dimensión mayor se normaliza a 100 mm para obtener un prototipo manejable.

In [ ]:
def preparar_curva(points, n=72, size_mm=100.0):
    clean=[points[0]]
    for p in points[1:]:
        if np.linalg.norm(p-clean[-1])>1e-6: clean.append(p)
    p=np.asarray(clean,float)
    if np.linalg.norm(p[0]-p[-1])>1e-6: p=np.vstack([p,p[0]])
    seg=np.linalg.norm(np.diff(p,axis=0),axis=1); s=np.r_[0,np.cumsum(seg)]
    if s[-1]<=1e-9: raise ValueError('La curva no tiene tamaño suficiente')
    q=np.linspace(0,s[-1],n,endpoint=False)
    out=np.column_stack([np.interp(q,s,p[:,0]),np.interp(q,s,p[:,1])])
    out-=out.mean(axis=0)
    out*=size_mm/max(np.ptp(out[:,0]),np.ptp(out[:,1]))
    return out

target=preparar_curva(curve_raw)
plt.figure(figsize=(7,5));plt.plot(*target.T,lw=3,label='Objetivo');plt.axis('equal');plt.grid(alpha=.25);plt.xlabel('x [mm]');plt.ylabel('y [mm]');plt.legend();plt.show()

## 3. Modelo, punto trazador y función de error

Variables de diseño: `x = [L1, L2, L3, L4, rP, phiP]`. El computador minimiza `RMS(x)` sujeto al cierre geométrico y a que **L2 complete 360° respecto a L1**.

- **A** es la articulación entre L2 y L3.
- **rP** es la distancia desde A hasta P.
- **phiP** es el ángulo constante medido desde la dirección positiva de L3 hasta AP.
- `AP = rP [cos(theta3 + phiP), sin(theta3 + phiP)]`.

La intersección de circunferencias puede dar una rama abierta y otra cruzada. El algoritmo conserva la solución más cercana a la posición anterior para impedir cambios de rama. La comparación final utiliza alineamiento rígido, desfase cíclico y ambos sentidos de recorrido.

In [ ]:
def intersecciones(c0,r0,c1,r1):
    v=c1-c0; d=np.linalg.norm(v)
    if d>r0+r1 or d<abs(r0-r1) or d<1e-10:return None
    a=(r0*r0-r1*r1+d*d)/(2*d); h=np.sqrt(max(0,r0*r0-a*a));m=c0+a*v/d;s=h*np.array([-v[1],v[0]])/d
    return np.array([m+s,m-s])

def simular(x,n=72):
    L1,L2,L3,L4,rP,phi=x;O2=np.array([0.,0.]);O4=np.array([L1,0.]);prev=None;poses=[]
    # Cierre analítico para todo theta2: L2 debe poder completar 360°.
    if abs(L1-L2) < abs(L3-L4) or L1+L2 > L3+L4:return None
    for th in np.linspace(0,2*np.pi,n,endpoint=False):
        A=L2*np.array([np.cos(th),np.sin(th)]);cand=intersecciones(A,L3,O4,L4)
        if cand is None:return None
        B=cand[np.argmax(cand[:,1])] if prev is None else cand[np.argmin(np.linalg.norm(cand-prev,axis=1))];prev=B
        u=(B-A)/L3;v=np.array([-u[1],u[0]]);P=A+rP*(np.cos(phi)*u+np.sin(phi)*v)
        poses.append((O2.copy(),A,B,O4.copy(),P))
    return poses

def alinear(P,Y):
    pm=P.mean(0);ym=Y.mean(0);X=P-pm;Z=Y-ym;H=X.T@Z
    ang=np.arctan2(H[0,1]-H[1,0],H[0,0]+H[1,1]);c,s=np.cos(ang),np.sin(ang);R=np.array([[c,-s],[s,c]])
    t=ym-pm@R.T;return P@R.T+t,R,t

def evaluar(x, devolver=False):
    poses=simular(x,len(target))
    if poses is None:return (1e6,None) if devolver else 1e6
    raw=np.asarray([p[4] for p in poses]);best=None
    for reverse in (False,True):
        base=raw[::-1] if reverse else raw
        for shift in range(len(target)):
            q=np.roll(base,-shift,axis=0);qa,R,t=alinear(q,target);e=np.linalg.norm(qa-target,axis=1)
            score=np.sqrt(np.mean(e*e))+.2*e.max()
            if best is None or score<best[0]:best=(score,qa,e,R,t,shift,reverse)
    return best if devolver else best[0]

bounds=[(35,150),(10,85),(25,150),(20,120),(3,120),(-np.pi,np.pi)]
print('Modelo listo.')

## 4. Síntesis inversa

La evolución diferencial hace la búsqueda global y Powell realiza el ajuste final. En Colab suele tardar entre uno y varios minutos. Para una búsqueda más profunda, aumenta `maxiter`.

In [ ]:
inicio=time.time()
de=differential_evolution(evaluar,bounds,maxiter=180,popsize=14,tol=1e-5,seed=21,polish=False,workers=1,updating='immediate',disp=True)
local=minimize(evaluar,de.x,method='Powell',bounds=bounds,options={'maxiter':450,'xtol':1e-5,'ftol':1e-6})
x=local.x if local.fun<de.fun else de.x
fit=evaluar(x,True);score,path,errors,R,t,shift,reverse=fit
L1,L2,L3,L4,rP,phiP=x
rms=float(np.sqrt(np.mean(errors**2)));max_error=float(errors.max());elapsed=time.time()-inicio
links=sorted([L1,L2,L3,L4]);grashof=links[0]+links[-1]<=links[1]+links[2]
print(f'RMS = {rms:.3f} mm | máximo = {max_error:.3f} mm | tiempo = {elapsed:.1f} s')

## 5. Resultados para construcción

Todas las longitudes se miden **entre centros de articulación**. `phiP` define la orientación del punto trazador respecto al acoplador L3.

In [ ]:
resultados=pd.DataFrame({'Magnitud':['L1 bancada','L2 manivela','L3 acoplador','L4 balancín','rP punto trazador','phiP'],'Valor':[L1,L2,L3,L4,rP,np.degrees(phiP)],'Unidad':['mm','mm','mm','mm','mm','grados']})
display(resultados.style.format({'Valor':'{:.3f}'}))
print('Condición general:', 'Grashof' if grashof else 'No Grashof')
print('Movilidad verificada: L2 completa 360° sin pérdida de cierre (restricción incluida en simular/evaluar).')
print(f'Márgenes de cierre: mínimo={abs(L1-L2)-abs(L3-L4):.3f} mm; máximo={L3+L4-(L1+L2):.3f} mm')
print(f'RMS: {rms:.3f} mm | Error máximo: {max_error:.3f} mm')
plt.figure(figsize=(8,6));plt.plot(*target.T,lw=4,label='Curva objetivo',alpha=.7);plt.plot(*path.T,lw=2.5,label='Cuatro barras');plt.axis('equal');plt.grid(alpha=.25);plt.xlabel('x [mm]');plt.ylabel('y [mm]');plt.legend();plt.title('Validación de trayectoria');plt.show()

## 6. Animación del mecanismo

In [ ]:
poses=simular(x,180)
if reverse: poses=poses[::-1]
poses=np.roll(np.asarray(poses,dtype=object),-(shift*180//len(target)),axis=0).tolist()
def transform(q):return np.asarray(q)@R.T+t
poses=[tuple(transform(q) for q in pose) for pose in poses]

fig,ax=plt.subplots(figsize=(8,6));ax.plot(*target.T,color='#457b9d',lw=4,alpha=.45,label='Objetivo');ax.plot(*path.T,color='#d84a8a',lw=2,label='Calculada');ax.set_aspect('equal');ax.grid(alpha=.2)
allp=np.vstack([np.vstack(p) for p in poses]+[target]);pad=15;ax.set_xlim(allp[:,0].min()-pad,allp[:,0].max()+pad);ax.set_ylim(allp[:,1].min()-pad,allp[:,1].max()+pad)
ground,=ax.plot([],[],lw=8,color='#64748b',solid_capstyle='round');crank,=ax.plot([],[],lw=7,color='#00b4d8');coupler,=ax.plot([],[],lw=7,color='#ff9f1c');rocker,=ax.plot([],[],lw=7,color='#2ec4b6');trace,=ax.plot([],[],lw=2,color='#d84a8a');joints=ax.scatter([],[],s=70,c='white',edgecolors='#12304a',zorder=5);point=ax.scatter([],[],s=100,c='#d84a8a',zorder=6)
def update(i):
    O2,A,B,O4,P=poses[i];ground.set_data([O2[0],O4[0]],[O2[1],O4[1]]);crank.set_data([O2[0],A[0]],[O2[1],A[1]]);coupler.set_data([A[0],B[0]],[A[1],B[1]]);rocker.set_data([B[0],O4[0]],[B[1],O4[1]]);trace.set_data([A[0],P[0]],[A[1],P[1]]);joints.set_offsets(np.vstack([O2,A,B,O4]));point.set_offsets(P.reshape(1,2));return ground,crank,coupler,rocker,trace,joints,point
ani=FuncAnimation(fig,update,frames=len(poses),interval=45,blit=True);plt.close(fig);display(HTML(ani.to_jshtml()))

## 7. Escala, puntos de validación y exportación

Selecciona la longitud máxima disponible del palito. Se exportan también las 12 posiciones teóricas correspondientes a `theta2 = 0°, 30°, ..., 330°`. En el prototipo se deben medir **esas mismas posiciones**, usando `O2=(0,0)`, el eje `x` en dirección `O2O4` y la misma escala.

In [ ]:
LONGITUD_UTIL_PALITO_MM=105.0
escala=min(1.0,LONGITUD_UTIL_PALITO_MM/max(L1,L2,L3,L4))
resultados['Valor_construido']=resultados['Valor']
resultados.loc[:4,'Valor_construido']=resultados.loc[:4,'Valor']*escala
resultados['Escala']=escala
resumen=pd.DataFrame({'RMS_mm':[rms],'Error_max_mm':[max_error],'Grashof':[grashof],'Tiempo_s':[elapsed]})
poses12=simular(x,12)
puntos12=np.asarray([pose[4]@R.T+t for pose in poses12])
validacion12=pd.DataFrame({'theta2_grados':np.arange(0,360,30),'x_teorico_mm':puntos12[:,0],'y_teorico_mm':puntos12[:,1]})
resultados.to_csv('geometria_4_barras.csv',index=False);resumen.to_csv('resumen_error.csv',index=False);validacion12.to_csv('puntos_teoricos_12_posiciones.csv',index=False)
print(f'Escala recomendada: {escala:.3f} (multiplicar las longitudes por este valor)')
display(resultados.style.format(precision=3))
display(validacion12.style.format(precision=3))
from google.colab import files
files.download('geometria_4_barras.csv')

## 8. Preguntas para el informe

1. Verifica la condición de Grashof con tus cuatro longitudes.
2. ¿Dónde aparece el mayor error entre las curvas y qué característica geométrica lo explica?
3. Compara las longitudes calculadas y construidas. ¿Cuál tiene mayor error porcentual?
4. ¿Cómo afectan la holgura de los remaches y el diámetro de las perforaciones?
5. Propón un cambio de geometría que mejore la trayectoria sin abandonar la familia de cuatro barras.